# Steel plant flexibility: cost and peak-demand impact under two rate designs

**Question:** how much do the bill and peak demand change if a steel facility sheds or shifts load, and how much of that depends on the rate design?

**Data:** `Steel_industry_data.csv` (15-minute kWh, one steel plant, 2018).

**Assumptions (change them in the cells below):**
- The plant is scaled by `SCALE` to represent a larger facility.
- *Shed* and *Shift* act only on weekdays 16:00-21:00 (a common grid peak window). Shed removes load (production lost). Shift moves the same share of window energy to off-peak hours of the same day, only into headroom below the baseline monthly peak.
- *Peak shave* caps each month's load at (1 - share) of its maximum, in any hour, and moves the clipped energy to same-day headroom under the cap. It is included because shed and shift only lower the billed peak if the monthly maximum falls in the window.
- The two tariffs are **illustrative rate designs, not real utility tariffs**. Swap in values from URDB for a specific utility.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

CSV = 'Steel_industry_data.csv'   # same folder as this notebook
SCALE = 10                        # assumption: ~100 kW plant scaled to a ~1 MW average facility
PEAK_START, PEAK_END = 16, 21     # weekday flexibility window, hours [start, end)

raw = pd.read_csv(CSV)
ts_end = pd.to_datetime(raw['date'], format='%d/%m/%Y %H:%M')
assert (ts_end.dt.day_name() == raw['Day_of_week']).mean() > 0.99, 'dates are not day-first'

# Timestamps are interval-ending, and 00:00 is the last interval of its own date (24:00).
# Roll those to the next day, then shift to interval-start so every row falls in 2018.
is_midnight = (ts_end.dt.hour == 0) & (ts_end.dt.minute == 0)
idx = ts_end.where(~is_midnight, ts_end + pd.Timedelta(days=1)) - pd.Timedelta(minutes=15)
load = pd.Series(raw['Usage_kWh'].to_numpy() * 4 * SCALE, index=idx, name='kW')  # kWh per 15 min -> kW
assert load.index.is_unique and load.index.is_monotonic_increasing

cal = pd.DataFrame(index=load.index)
cal['month'] = load.index.month
cal['summer'] = cal['month'].between(6, 9)
cal['weekday'] = load.index.dayofweek < 5
cal['peak'] = cal['weekday'] & (load.index.hour >= PEAK_START) & (load.index.hour < PEAK_END)
cal['date'] = load.index.normalize()

print(f'{len(load):,} intervals, {load.index.min()} to {load.index.max()}')
print(f'average {load.mean()/1000:.2f} MW, peak {load.max()/1000:.2f} MW, annual {load.sum()*0.25/1e6:.1f} GWh')

In [ ]:
# Illustrative rate designs; replace with URDB values for a real utility
TARIFFS = {
    'A: TOU energy + peak demand': dict(
        energy=dict(sp=0.28, so=0.11, wp=0.15, wo=0.09),  # $/kWh: summer/winter x peak/off-peak
        demand_all=8.0,      # $/kW-month on the monthly max (all hours)
        demand_peak=14.0,    # $/kW-month on the summer peak-window max
        fixed=500.0),        # $/month
    'B: flat energy + high demand charge': dict(
        energy=dict(sp=0.055, so=0.055, wp=0.055, wo=0.055),
        demand_all=16.0, demand_peak=0.0, fixed=500.0),
}

def bill(kw, t):
    e = t['energy']
    rate = np.where(cal['summer'],
                    np.where(cal['peak'], e['sp'], e['so']),
                    np.where(cal['peak'], e['wp'], e['wo']))
    energy = float((kw * 0.25 * rate).sum())
    d_all = kw.groupby(cal['month']).max().sum() * t['demand_all']
    sp = cal['summer'] & cal['peak']
    d_peak = kw[sp].groupby(cal['month'][sp]).max().sum() * t['demand_peak']
    return dict(energy=energy, demand=float(d_all + d_peak), fixed=12 * t['fixed'])

def shed(kw, frac):
    out = kw.copy()
    out[cal['peak']] = kw[cal['peak']] * (1 - frac)
    return out

def shift(kw, frac):
    # Headroom is capped at the baseline monthly max so the shift cannot create a new peak
    peak = cal['peak']
    out = kw.copy()
    moved_kwh = (kw[peak] * frac * 0.25).groupby(cal['date'][peak]).sum()
    out[peak] = kw[peak] * (1 - frac)
    cap = kw.groupby(cal['month']).transform('max')
    room = (cap - out).clip(lower=0).where(~peak, 0.0)
    room_kwh_day = (room * 0.25).groupby(cal['date']).sum()
    share = (moved_kwh / room_kwh_day).fillna(0).clip(upper=1)
    return out + room * cal['date'].map(share).fillna(0)

def shave(kw, frac):
    # Clip each month at (1 - frac) x its max; move clipped energy into same-day headroom under that cap
    cap = kw.groupby(cal['month']).transform('max') * (1 - frac)
    out = kw.clip(upper=cap)
    clipped_kwh = ((kw - out) * 0.25).groupby(cal['date']).sum()
    room = (cap - out).clip(lower=0)
    room_kwh_day = (room * 0.25).groupby(cal['date']).sum()
    share = (clipped_kwh / room_kwh_day).fillna(0).clip(upper=1)
    return out + room * cal['date'].map(share).fillna(0)

def evaluate(kw_new, tname):
    t = TARIFFS[tname]
    b0, b1 = bill(load, t), bill(kw_new, t)
    tot0, tot1 = sum(b0.values()), sum(b1.values())
    return dict(tariff=tname,
                energy_sav=b0['energy'] - b1['energy'],
                demand_sav=b0['demand'] - b1['demand'],
                total_sav=tot0 - tot1,
                sav_pct=100 * (tot0 - tot1) / tot0)

def peak_stats(kw):
    m = kw.groupby(cal['month']).max()
    return dict(annual_peak_kw=m.max(), avg_monthly_peak_kw=m.mean())

In [ ]:
FRAC = 0.15
scen = {f'Shed {FRAC:.0%}': shed(load, FRAC),
        f'Shift {FRAC:.0%}': shift(load, FRAC),
        f'Peak shave {FRAC:.0%}': shave(load, FRAC)}

# Share of removed energy that found headroom (the rest is effectively shed)
moved = (load[cal['peak']] * FRAC * 0.25).sum()
recovered = (scen[f'Shift {FRAC:.0%}'].sum() - load.sum()) * 0.25 + moved
print(f'shift: {100 * recovered / moved:.1f}% of moved energy recovered off-peak')
clipped = (load - load.clip(upper=load.groupby(cal['month']).transform('max') * (1 - FRAC))).sum() * 0.25
recovered = (scen[f'Peak shave {FRAC:.0%}'].sum() - load.sum()) * 0.25 + clipped
print(f'peak shave: {100 * recovered / clipped:.1f}% of clipped energy recovered')

pk_when = load.groupby(cal['month']).idxmax()
n_in_window = int(cal.loc[pk_when.to_numpy(), 'peak'].sum())
print(f'baseline monthly peaks inside the flex window: {n_in_window} of 12')

rows = []
for sname, kw in scen.items():
    for tname in TARIFFS:
        r = evaluate(kw, tname)
        r['scenario'] = sname
        rows.append(r)
res = pd.DataFrame(rows).set_index(['scenario', 'tariff'])

base = pd.DataFrame({tn: bill(load, t) for tn, t in TARIFFS.items()}).T
base['total'] = base.sum(axis=1)
peaks = pd.DataFrame({'Baseline': peak_stats(load), **{k: peak_stats(v) for k, v in scen.items()}}).T
peaks['avg_monthly_peak_change_%'] = 100 * (peaks['avg_monthly_peak_kw'] / peaks.loc['Baseline', 'avg_monthly_peak_kw'] - 1)

print('Baseline annual bill ($)')
display(base.round(0))
print('Savings vs baseline ($/yr)')
display(res.round(1))
print('Peak demand (kW)')
display(peaks.round(1))

## Plot 1: what the strategies do to the load (summer week)

In [ ]:
wk = slice('2018-07-09', '2018-07-15 23:45')   # Monday to Sunday
fig, ax = plt.subplots(figsize=(11, 4))
for label, s in {'Baseline': load, **scen}.items():
    ax.plot(s[wk].index, s[wk].values / 1000, label=label, lw=1)
for i, day in enumerate(pd.date_range('2018-07-09', periods=5)):
    ax.axvspan(day + pd.Timedelta(hours=PEAK_START), day + pd.Timedelta(hours=PEAK_END),
               color='orange', alpha=0.15, label='Peak window' if i == 0 else None)
ax.set_ylabel('Load (MW)')
ax.set_title('Plant load, week of 9 July 2018')
ax.legend(ncol=5, loc='upper right')
plt.tight_layout()
plt.show()

## Plot 2: where the savings come from, by tariff

In [ ]:
x = np.arange(len(res))
labels = [f'{s}\nTariff {t[0]}' for s, t in res.index]
top = max(res['energy_sav'].max(), res['demand_sav'].max()) / 1e3
fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(x - 0.2, res['energy_sav'] / 1e3, 0.4, label='Energy charges')
ax.bar(x + 0.2, res['demand_sav'] / 1e3, 0.4, label='Demand charges')
for xi, (tot, pct) in enumerate(zip(res['total_sav'], res['sav_pct'])):
    ax.text(xi, top * 1.05, f'${tot/1e3:,.0f}k ({pct:.1f}%)', ha='center', fontsize=9)
ax.set_ylim(min(0, res[['energy_sav', 'demand_sav']].min().min() / 1e3 * 1.1), top * 1.2)
ax.axhline(0, color='k', lw=0.8)
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylabel('Annual savings ($ thousand)')
ax.set_title('Savings by bill component (total and % of bill above bars)')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

## Plot 3: sensitivity to how much load is flexed

In [ ]:
base_billed = load.groupby(cal['month']).max().sum()
sweep = []
for f in np.arange(0, 0.31, 0.05):
    for sname, fn in (('Shed', shed), ('Shift', shift), ('Peak shave', shave)):
        kw = fn(load, f)
        row = dict(frac=100 * f, strategy=sname,
                   peak_red_pct=100 * (1 - kw.groupby(cal['month']).max().sum() / base_billed))
        for tname in TARIFFS:
            row[tname] = evaluate(kw, tname)['sav_pct']
        sweep.append(row)
sweep = pd.DataFrame(sweep)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))
colors = dict(zip(TARIFFS, ['tab:blue', 'tab:red']))
styles = {'Shed': '-', 'Shift': '--', 'Peak shave': ':'}
for tname in TARIFFS:
    for sname in styles:
        d = sweep[sweep['strategy'] == sname]
        ax1.plot(d['frac'], d[tname], styles[sname], color=colors[tname], marker='o', label=f'{sname}, tariff {tname[0]}')
ax1.set_xlabel('Share of load flexed (%)')
ax1.set_ylabel('Annual bill savings (%)')
ax1.set_title('Bill savings depend on the rate design')
ax1.legend(fontsize=8)
for sname in styles:
    d = sweep[sweep['strategy'] == sname]
    ax2.plot(d['frac'], d['peak_red_pct'], styles[sname], color='k', marker='o', label=sname)
ax2.set_xlabel('Share of load flexed (%)')
ax2.set_ylabel('Reduction in billed monthly peak (%)')
ax2.set_title('Peak demand change (same under both tariffs)')
ax2.legend()
plt.tight_layout()
plt.show()

In [ ]:
for (s, t), r in res.iterrows():
    print(f'{s:16s} under tariff {t[0]}: saves ${r.total_sav:,.0f}/yr ({r.sav_pct:.1f}% of bill)')
print()
print(peaks[['annual_peak_kw', 'avg_monthly_peak_kw', 'avg_monthly_peak_change_%']].round(1))

## How to read the results

- **Peak demand** is a property of the load, so it changes identically under both tariffs. Shed and shift only lower it if the monthly maximum falls inside the flexibility window (see the count printed above); peak shave lowers it directly. Plot 3 (right) shows this.
- **Cost** differs by rate design. A TOU energy tariff rewards moving energy out of the window even when the monthly peak is unchanged. A flat-energy, high-demand-charge tariff pays only for lowering the monthly maximum, and shedding saves energy cost where shifting does not.
- The tariffs, the scale factor and the flex percentages are assumptions. Treat the output as a method demonstration, not a forecast for any real plant.